In [0]:
# Confirm our Unity Catalog volume is accessible
volume_path = "/Volumes/workspace/default/cyclistic_data"

# Create subfolders for each medallion layer
import os
for folder in ["raw", "bronze", "silver", "gold"]:
    dbutils.fs.mkdirs(f"{volume_path}/{folder}")
    print(f"Created: {volume_path}/{folder}")

print("\nAll folders ready!")

In [0]:
import requests
import zipfile
import xml.etree.ElementTree as ET
from io import BytesIO

BUCKET_ROOT = "https://divvy-tripdata.s3.amazonaws.com"
LIST_URL = BUCKET_ROOT + "?list-type=2&max-keys=1000"
RAW_PATH = "/Volumes/workspace/default/cyclistic_data/raw"

# Set up a session with retries
from requests.adapters import HTTPAdapter, Retry
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=0.5,
    status_forcelist=(429, 500, 502, 503, 504)
)))
session.headers.update({"User-Agent": "divvy-downloader/1.0"})

# Find all zip files in the bucket
def list_zip_keys():
    keys, next_token = [], None
    while True:
        url = LIST_URL if not next_token else f"{LIST_URL}&continuation-token={next_token}"
        root = ET.fromstring(session.get(url, timeout=60).content)
        ns = {"s3": root.tag.split("}")[0].strip("{")} if "}" in root.tag else {}
        for el in root.findall("s3:Contents/s3:Key", ns) or root.findall("Contents/Key"):
            if el.text.strip().lower().endswith(".zip"):
                keys.append(el.text.strip())
        trunc = root.find("s3:IsTruncated", ns) or root.find("IsTruncated")
        if trunc is not None and trunc.text.lower() == "true":
            nt = root.find("s3:NextContinuationToken", ns) or root.find("NextContinuationToken")
            next_token = nt.text
        else:
            break
    return keys

# Only keep Oct 2024 to Sep 2025 (our 12 months)
TARGET_MONTHS = [
    "202410", "202411", "202412",
    "202501", "202502", "202503",
    "202504", "202505", "202506",
    "202507", "202508", "202509"
]

print("Finding files in S3 bucket...")
all_keys = list_zip_keys()
our_keys = [k for k in all_keys if any(m in k for m in TARGET_MONTHS)]
print(f"Found {len(our_keys)} matching files\n")

# Download and extract each one
for i, key in enumerate(sorted(our_keys), 1):
    fname = key.split("/")[-1]
    print(f"[{i}/{len(our_keys)}] Downloading {fname}...")
    try:
        resp = session.get(f"{BUCKET_ROOT}/{key}", timeout=300)
        resp.raise_for_status()
        with zipfile.ZipFile(BytesIO(resp.content)) as zf:
            for name in zf.namelist():
                if name.endswith(".csv"):
                    data = zf.read(name)
                    out_path = f"{RAW_PATH}/{name}"
                    with open(out_path, "wb") as f:
                        f.write(data)
                    print(f"   Saved: {name}")
    except Exception as e:
        print(f"   ERROR: {e}")

print("\nAll done! Raw CSV files are in your volume.")

In [0]:
# List all files in our raw folder and show their sizes
files = dbutils.fs.ls(f"{RAW_PATH}/")

print(f"Total files found: {len(files)}\n")
print(f"{'Filename':<50} {'Size (MB)':>10}")
print("-" * 62)

total_size = 0
for f in sorted(files, key=lambda x: x.name):
    size_mb = f.size / (1024 * 1024)
    total_size += size_mb
    print(f"{f.name:<50} {size_mb:>9.1f} MB")

print("-" * 62)
print(f"{'TOTAL':<50} {total_size:>9.1f} MB")